In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT5 project root (paths.py). Run Jupyter with cwd GPT5 or GPT5/notebooks."
    )
import paths

In [3]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.RELEVANCY / "gpt5-relevancy-combined-dec-12.csv"
df = pd.read_csv(output_path)

# # Extract the predicted letter from the format <answer>Option A</answer>
# def extract_letter_from_xml(pred):
#     if isinstance(pred, str):
#         match = re.search(r"<answer>Option\s+([A-J])</answer>", pred, re.IGNORECASE)
#         if match:
#             return match.group(1).upper()
#     return None

# df["gpt_letter"] = df["majority_vote"].apply(extract_letter_from_xml)

# # Clean and standardize the ground truth answer
# df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["majority_vote"] == row["answer_corr"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


Letter-Based Correct Predictions: 1118
Total Predictions Compared: 1300
Letter Match Accuracy: 86.00%
Data Source: jama
  Correct Predictions: 514
  Total Predictions: 582
  Accuracy: 88.32%
  Std Dev: 0.3215

Data Source: medxpert
  Correct Predictions: 220
  Total Predictions: 318
  Accuracy: 69.18%
  Std Dev: 0.4625

Data Source: medbullets
  Correct Predictions: 194
  Total Predictions: 207
  Accuracy: 93.72%
  Std Dev: 0.2432

Data Source: mmlu
  Correct Predictions: 190
  Total Predictions: 193
  Accuracy: 98.45%
  Std Dev: 0.1240



In [4]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")

Standard Deviation of Accuracy Across Data Sources: 0.1284


## Removed GPT-5 Self Reported Low+Irr Sentences

In [1]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.RELEVANCY / "gpt5-irr-removed-relevancy-combined-dec-12.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the format <answer>Option A</answer>
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"<answer>Option\s+([A-J])</answer>", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["majority_vote"].apply(extract_letter_from_xml)

# Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


Letter-Based Correct Predictions: 1095
Total Predictions Compared: 1300
Letter Match Accuracy: 84.23%
Data Source: jama
  Correct Predictions: 504
  Total Predictions: 582
  Accuracy: 86.60%
  Std Dev: 0.3410

Data Source: medxpert
  Correct Predictions: 213
  Total Predictions: 318
  Accuracy: 66.98%
  Std Dev: 0.4710

Data Source: medbullets
  Correct Predictions: 189
  Total Predictions: 207
  Accuracy: 91.30%
  Std Dev: 0.2825

Data Source: mmlu
  Correct Predictions: 189
  Total Predictions: 193
  Accuracy: 97.93%
  Std Dev: 0.1428



In [2]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")

Standard Deviation of Accuracy Across Data Sources: 0.1332


In [ ]:
# --- Physician Origin subset recalculation ---
from pathlib import Path
import pandas as pd
import re
import numpy as np

physician_csv = Path(r"/home/yuexing/NeuRIPS25/Physician_Labels/Mar2_2026_Data/933_Clinician_Student_Majority_Vote.csv")
physician_origins = set(
    pd.read_csv(physician_csv, usecols=['Origin'])['Origin'].astype(str).str.strip()
)

if 'df' in locals() and isinstance(df, pd.DataFrame):
    df_eval = df.copy()
elif 'output_path' in locals():
    df_eval = pd.read_csv(output_path)
else:
    raise RuntimeError('Could not find dataframe `df` or `output_path` in this notebook state.')

if 'Origin' not in df_eval.columns:
    raise KeyError('`Origin` column is missing from evaluation dataframe.')

df_eval = df_eval[df_eval['Origin'].astype(str).str.strip().isin(physician_origins)].copy()
print(f"Physician-Origin subset rows: {len(df_eval)}")

if len(df_eval) == 0:
    raise ValueError('No overlapping Origin IDs found with physician CSV.')

# Build gpt_letter when not already present
if 'gpt_letter' not in df_eval.columns:
    pred_col = None
    for c in ['gpt5_direct_prediction', 'gpt4o_direct_prediction', 'majority_vote', 'GPT5_on_72B_SR']:
        if c in df_eval.columns:
            pred_col = c
            break
    if pred_col is None:
        raise KeyError('No supported prediction column found to derive `gpt_letter`.')

    def extract_letter(x):
        if not isinstance(x, str):
            return None
        m = re.search(r'Option\s*\[?([A-J])\]?|^\s*([A-J])\s*$', str(x).strip(), flags=re.IGNORECASE)
        if m:
            return (m.group(1) or m.group(2)).upper()
        return None

    if pred_col == 'GPT5_on_72B_SR':
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    elif pred_col == 'majority_vote' and 'answer_corr' in df_eval.columns:
        # For this notebook style majority_vote is often already a letter
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    else:
        df_eval['gpt_letter'] = df_eval[pred_col].apply(extract_letter)

# Build answer_letter
if 'answer_letter' not in df_eval.columns:
    if 'answer_corr' in df_eval.columns:
        df_eval['answer_letter'] = df_eval['answer_corr'].astype(str).str.strip().str.upper()
    else:
        raise KeyError('No `answer_corr` column available to build `answer_letter`.')

# Match + binary columns
if 'gpt_letter_match' not in df_eval.columns:
    df_eval['gpt_letter_match'] = np.where(
        df_eval['gpt_letter'] == df_eval['answer_letter'],
        'Correct',
        'Incorrect'
    )

df_eval['gpt_letter_binary'] = (df_eval['gpt_letter_match'] == 'Correct').astype(int)

correct_count = int(df_eval['gpt_letter_binary'].sum())
total_count = int(df_eval['gpt_letter_binary'].notna().sum())
accuracy = correct_count / total_count if total_count > 0 else 0.0

print('\n=== Physician-Origin Recalculation ===')
print(f"Correct Predictions: {correct_count}")
print(f"Total Predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")

source_col = None
for c in ['data_source_df3', 'data_source_corr', 'data_source_corr_trainee']:
    if c in df_eval.columns:
        source_col = c
        break

if source_col:
    print(f"\nPer-data-source stats ({source_col}):")
    for source in df_eval[source_col].dropna().unique():
        source_df = df_eval[df_eval[source_col] == source]
        c = int(source_df['gpt_letter_binary'].sum())
        t = int(source_df['gpt_letter_binary'].notna().sum())
        acc = c / t if t > 0 else 0.0
        std = source_df['gpt_letter_binary'].std(ddof=1) if t > 1 else float('nan')
        print(f"  {source}: Correct={c}, Total={t}, Accuracy={acc:.2%}, Std={std:.4f}")
